In [1]:
import pandas as pd
df = pd.read_csv(r'/root/autodl-tmp/CommitFit/dataset/Ghadhab/dataset.csv')
df

,user,repo,commit,labels,msgs,diffs,feature
0,ponsonio,RxJava,0531b8bff5c14d9504beefb4ad47f473e3a22932,Perfective,Change hasException to hasThrowable--,diff --git a/rxjava-core/src/main/java/rx/Noti...,"[1, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,ponsonio,RxJava,0950c46beda335819928585f1262dfe1dca78a0b,Adaptive,Trying to extend the Scheduler interface accor...,diff --git a/rxjava-core/src/main/java/rx/Sche...,"[2, 44, 0, 0, 30, 0, 0, 1, 18, 0, 0, 0, 0, 0, ..."
2,ponsonio,RxJava,0f92fdd8e6422d5b79c610a7fd8409d222315a49,Adaptive,RunAsync method for outputting multiple values--,diff --git a/rxjava-contrib/rxjava-async-util/...,"[2, 53, 0, 0, 42, 0, 0, 1, 45, 1, 0, 0, 0, 0, ..."
3,ponsonio,RxJava,100f571c9a2835d5a30a55374b9be74c147e031f,Corrective,forEach with Action1 but not Observer--I re-re...,diff --git a/language-adaptors/rxjava-groovy/s...,"[1, 5, 122, 9, 10, 9, 4, 1, 5, 18, 2, 0, 0, 0,..."
4,ponsonio,RxJava,191f023cf5253ea90647bc091dcaf55ccdce81cc,Corrective,1.x: Fix Completable swallows- OnErrorNotImple...,diff --git a/src/main/java/rx/Completable.java...,"[1, 1, 0, 0, 0, 0, 0, 1, 21, 0, 0, 0, 0, 0, 0,..."
...,...,...,...,...,...,...,...
1776,jenkinsci,clearcase-plugin,51e9da224f80254476a7dc446bca817b505381d8,Perfective,Use a temporary file to decrease memory consum...,diff --git a/src/main/java/hudson/plugins/clea...,"[2, 12, 0, 4, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,..."
1777,jexp,batch-import,609d6c4b1eea2c33d9fb950fcbb9ba9dc1f80fc3,Perfective,added a more memory efficient structure for st...,diff --git a/src/main/java/org/neo4j/batchimpo...,"[10, 159, 29, 35, 9, 2, 1, 5, 106, 0, 4, 8, 0,..."
1778,hdiv,hdiv,19b650c78a1c76f4fd90274d7f163f863c0d39e4,Perfective,Memory and performance optimizations,diff --git a/hdiv-config/src/main/java/org/hdi...,"[31, 302, 131, 140, 170, 89, 53, 7, 88, 14, 17..."
1779,casidiablo,persistence,d7bf95159df37a3d338ca267dddd3d26b38ec37c,Perfective,Now it is possible to specify the sqlite open ...,diff --git a/pom.xml b/pom.xml\nindex 394263b....,"[5, 57, 20, 9, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [2]:
import re
from typing import Literal

# -----------------------------
# Patterns: lines to drop
# -----------------------------
RE_DROP_LINE = re.compile(
    r"(git-svn-id:|git-p4:|svn\.apache\.org|cherry picked from commit)",
    re.IGNORECASE,
)
RE_ONLY_DASHES = re.compile(r"^\s*-{2,}\s*$")
RE_TOOL_BRACKET = re.compile(r"^\s*\[git-p4:.*\]\s*$", re.IGNORECASE)

# -----------------------------
# Subject-level cleanup
# -----------------------------
RE_URL = re.compile(r"\bhttps?://\S+|\bwww\.\S+", re.IGNORECASE)
RE_MISSING_SPACE_BEFORE_QUOTE = re.compile(r"(\b\w+)'(?=\w)")
RE_DOUBLE_DASH = re.compile(r"\s*--+\s*")
RE_HYPHEN_WRAP = re.compile(r"(\w)\s*-\s+(?=\w)")
RE_TRAILING_DASHES = re.compile(r"[-–—]{2,}\s*$")

# Issue / PR / ticket markers that often cannot be inferred from diff
# Covers:
#  - [ABC-123], ABC-123
#  - Issue 123 / Issue -123 / issue: 123
#  - Fix #123 / Closes #123 / Resolves #123 / (#123) / PR #123
RE_ISSUE = re.compile(
    r"""
    \[[A-Z][A-Z0-9]+-\d+\]                      # [ABC-123]
    |\b[A-Z][A-Z0-9]+-\d+\b                     # ABC-123
    |\b[Ii]ssue\s*[:#]?\s*-?\s*\d+\b            # Issue 123 / Issue -123 / Issue:123
    |\b(?:[Ff]ix(?:es)?|[Cc]lose(?:s)?|[Rr]esolve(?:s)?)\s+#\d+\b  # Fixes #123
    |\b(?:PR|Pull\s+Request)\s*#\d+\b           # PR #123 / Pull Request #123
    |\(#\d+\)                                   # (#123)
    """,
    re.VERBOSE,
)

# Optional: remove commit hashes and revision identifiers
RE_HASH = re.compile(r"\b[0-9a-f]{7,40}\b", re.IGNORECASE)

# Optional: strip leading "Revert" prefix (often noisy for generation)
RE_REVERT_PREFIX = re.compile(r"^\s*revert\s+\"?(.*?)\"?\s*$", re.IGNORECASE)

# -----------------------------
# Merge detection
# -----------------------------
RE_MERGE_SUBJECT = re.compile(
    r"^\s*merge\s+(branch|remote-tracking\s+branch|pull\s+request)\b",
    re.IGNORECASE,
)


def _normalize_subject(subject: str, drop_issue_ids: bool = True, drop_hashes: bool = True) -> str:
    """Normalize a single subject line (title line) to reduce noisy variance."""
    s = subject.strip()

    # Remove URLs (often irrelevant for semantic label + hurts ROUGE)
    s = RE_URL.sub("", s).strip()

    # Drop ticket/PR markers that are hard to predict from diff
    if drop_issue_ids:
        s = RE_ISSUE.sub("", s).strip()

    # Drop raw commit hashes
    if drop_hashes:
        s = RE_HASH.sub("", s).strip()

    # Fix missing space before apostrophe: "dont'care" -> "dont 'care"
    s = RE_MISSING_SPACE_BEFORE_QUOTE.sub(r"\1 '", s)

    # Collapse multiple dashes into a single space: "foo -- bar" -> "foo bar"
    s = RE_DOUBLE_DASH.sub(" ", s)

    # Fix hyphen-wrapped tokens caused by line breaks: "a- subclass" -> "a subclass"
    s = RE_HYPHEN_WRAP.sub(r"\1 ", s)

    # Strip trailing long dash sequences
    s = RE_TRAILING_DASHES.sub("", s).strip()

    # Remove leftover leading punctuation from stripped IDs (":", "-", etc.)
    s = re.sub(r"^[\s:;,\-–—]+", "", s).strip()

    # Normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _filter_noise_lines(text: str) -> str:
    """Drop tool/metadata/noise lines while preserving blank lines."""
    kept = []
    for raw in text.split("\n"):
        stripped = raw.strip()

        # Keep blank lines to preserve paragraph separation
        if not stripped:
            kept.append("")
            continue

        # Drop obvious separators / tool markers / SCM metadata
        if RE_ONLY_DASHES.match(stripped):
            continue
        if RE_TOOL_BRACKET.match(stripped):
            continue
        if RE_DROP_LINE.search(stripped):
            continue

        kept.append(raw)

    return "\n".join(kept).strip()


def _extract_subject(text: str) -> str:
    """Extract the subject (first non-empty line of the first paragraph)."""
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    if not paras:
        return ""

    subject = paras[0].split("\n", 1)[0].strip()
    subject = re.sub(r"\s+", " ", subject).strip()
    return subject


def clean_commit_message_subject(
    msg: str,
    merge_policy: Literal["drop", "merge", "keep"] = "drop",
    drop_issue_ids: bool = True,
    drop_hashes: bool = True,
    drop_reverts: bool = False,
) -> str:
    """
    Clean and normalize a commit message subject line.

    merge_policy:
      - "drop": remove merge commits entirely (return "")
      - "merge": collapse any merge subject to the token "Merge"
      - "keep": keep merge subject after normalization
    """
    if not isinstance(msg, str) or not msg.strip():
        return ""

    # Normalize newlines
    text = msg.replace("\r\n", "\n").replace("\r", "\n")

    # Remove noisy metadata/tool lines
    text = _filter_noise_lines(text)
    if not text:
        return ""

    # Extract subject
    subject = _extract_subject(text)
    if not subject:
        return ""

    # Optionally drop revert messages (or normalize them)
    if drop_reverts and RE_REVERT_PREFIX.match(subject):
        return ""

    # Normalize subject (URLs, issue IDs, hashes, punctuation, whitespace)
    subject = _normalize_subject(subject, drop_issue_ids=drop_issue_ids, drop_hashes=drop_hashes)
    if not subject:
        return ""

    # Merge handling
    if RE_MERGE_SUBJECT.match(subject):
        if merge_policy == "drop":
            return ""
        if merge_policy == "merge":
            return "Merge"
        # "keep": return normalized subject

    return subject

In [3]:
df["gt_clean"] = df["msgs"].fillna("").map(clean_commit_message_subject)


In [4]:
df

,user,repo,commit,labels,msgs,diffs,feature,gt_clean
0,ponsonio,RxJava,0531b8bff5c14d9504beefb4ad47f473e3a22932,Perfective,Change hasException to hasThrowable--,diff --git a/rxjava-core/src/main/java/rx/Noti...,"[1, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Change hasException to hasThrowable
1,ponsonio,RxJava,0950c46beda335819928585f1262dfe1dca78a0b,Adaptive,Trying to extend the Scheduler interface accor...,diff --git a/rxjava-core/src/main/java/rx/Sche...,"[2, 44, 0, 0, 30, 0, 0, 1, 18, 0, 0, 0, 0, 0, ...",Trying to extend the Scheduler interface accor...
2,ponsonio,RxJava,0f92fdd8e6422d5b79c610a7fd8409d222315a49,Adaptive,RunAsync method for outputting multiple values--,diff --git a/rxjava-contrib/rxjava-async-util/...,"[2, 53, 0, 0, 42, 0, 0, 1, 45, 1, 0, 0, 0, 0, ...",RunAsync method for outputting multiple values
3,ponsonio,RxJava,100f571c9a2835d5a30a55374b9be74c147e031f,Corrective,forEach with Action1 but not Observer--I re-re...,diff --git a/language-adaptors/rxjava-groovy/s...,"[1, 5, 122, 9, 10, 9, 4, 1, 5, 18, 2, 0, 0, 0,...",forEach with Action1 but not Observer I re-rea...
4,ponsonio,RxJava,191f023cf5253ea90647bc091dcaf55ccdce81cc,Corrective,1.x: Fix Completable swallows- OnErrorNotImple...,diff --git a/src/main/java/rx/Completable.java...,"[1, 1, 0, 0, 0, 0, 0, 1, 21, 0, 0, 0, 0, 0, 0,...",1.x: Completable swallows OnErrorNotImplemente...
...,...,...,...,...,...,...,...,...
1776,jenkinsci,clearcase-plugin,51e9da224f80254476a7dc446bca817b505381d8,Perfective,Use a temporary file to decrease memory consum...,diff --git a/src/main/java/hudson/plugins/clea...,"[2, 12, 0, 4, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,...",Use a temporary file to decrease memory consum...
1777,jexp,batch-import,609d6c4b1eea2c33d9fb950fcbb9ba9dc1f80fc3,Perfective,added a more memory efficient structure for st...,diff --git a/src/main/java/org/neo4j/batchimpo...,"[10, 159, 29, 35, 9, 2, 1, 5, 106, 0, 4, 8, 0,...",added a more memory efficient structure for st...
1778,hdiv,hdiv,19b650c78a1c76f4fd90274d7f163f863c0d39e4,Perfective,Memory and performance optimizations,diff --git a/hdiv-config/src/main/java/org/hdi...,"[31, 302, 131, 140, 170, 89, 53, 7, 88, 14, 17...",Memory and performance optimizations
1779,casidiablo,persistence,d7bf95159df37a3d338ca267dddd3d26b38ec37c,Perfective,Now it is possible to specify the sqlite open ...,diff --git a/pom.xml b/pom.xml\nindex 394263b....,"[5, 57, 20, 9, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Now it is possible to specify the sqlite open ...


In [5]:
df.to_csv('Ghadhab_dataset.csv',index=0,encoding='utf_8_sig')

In [6]:
from core_diff_extractor import add_lightweight_changescribe_columns, lightweight_changescribe
import pandas as pd
# CSV_PATH = "/root/autodl-tmp/CommitFit/dataset/Ghadhab/dataset.csv"

# # =========================
# # 2) 读 CSV -> Dataset -> split
# # =========================
# df = pd.read_csv(CSV_PATH)




In [7]:
sample = r"""diff --git a/rxjava-core/src/main/java/rx/Notification.java b/rxjava-core/src/main/java/rx/Notification.java
index ad6b81c0..866ed064 100644
--- a/rxjava-core/src/main/java/rx/Notification.java
+++ b/rxjava-core/src/main/java/rx/Notification.java
@@ -92,5 +92,5 @@ public class Notification<T> {
-    public boolean hasException() {
+    public boolean hasThrowable() {
     return isOnError() && throwable != null;
 }
@@ -126,5 +126,5 @@ public class Notification<T> {
-        if (hasException())
+        if (hasThrowable())
         str.append(" ").append(getThrowable().getMessage());
     str.append("]");
@@ -137,5 +137,5 @@ public class Notification<T> {
-        if (hasException())
+        if (hasThrowable())
         hash = hash * 31 + getThrowable().hashCode();
     return hash;
@@ -155,5 +155,5 @@ public class Notification<T> {
-        if (hasException() && !getThrowable().equals(notification.getThrowable()))
+        if (hasThrowable() && !getThrowable().equals(notification.getThrowable()))
         return false;
     return true;
"""
print(lightweight_changescribe(sample, repo_name="rxjava", change_type="Refactor"))

rxjava [Refactor] ChangeScribeStart
Summarized Code Changes:
File: java/rx/Notification.java
 - Rename identifier `hasException` to `hasThrowable` (1 occurrence) in java/rx/Notification.java
 - Modify conditional expression from `hasException()` to `hasThrowable()` in java/rx/Notification.java
 - Modify conditional expression from `hasException() && !getThrowable().equals(notification.getThrowable())` to `hasThrowable()` in java/rx/Notification.java
 - Remove method call `hasException()` in java/rx/Notification.java
 - Add method call `hasThrowable()` in java/rx/Notification.java
End change part


In [8]:
add_lightweight_changescribe_columns(
    in_csv="Ghadhab_dataset.csv",
    out_csv="data.with_changescribe.csv",
    code_diff_col="diffs",
    repo_col='repo',          # 如果你 CSV 有 repo 列就填，比如 "repo"
    change_type_col=None,   # 如果你 CSV 有 change_type 列就填，比如 "label"
)

[OK] Wrote: data.with_changescribe.csv


In [9]:
df = pd.read_csv('data.with_changescribe.csv')
df

,user,repo,commit,labels,msgs,diffs,feature,gt_clean,changescribe_text,core_diff
0,ponsonio,RxJava,0531b8bff5c14d9504beefb4ad47f473e3a22932,Perfective,Change hasException to hasThrowable--,diff --git a/rxjava-core/src/main/java/rx/Noti...,"[1, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Change hasException to hasThrowable,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: java/rx/Notifi...
1,ponsonio,RxJava,0950c46beda335819928585f1262dfe1dca78a0b,Adaptive,Trying to extend the Scheduler interface accor...,diff --git a/rxjava-core/src/main/java/rx/Sche...,"[2, 44, 0, 0, 30, 0, 0, 1, 18, 0, 0, 0, 0, 0, ...",Trying to extend the Scheduler interface accor...,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: java/rx/Schedu...
2,ponsonio,RxJava,0f92fdd8e6422d5b79c610a7fd8409d222315a49,Adaptive,RunAsync method for outputting multiple values--,diff --git a/rxjava-contrib/rxjava-async-util/...,"[2, 53, 0, 0, 42, 0, 0, 1, 45, 1, 0, 0, 0, 0, ...",RunAsync method for outputting multiple values,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: util/async/Asy...
3,ponsonio,RxJava,100f571c9a2835d5a30a55374b9be74c147e031f,Corrective,forEach with Action1 but not Observer--I re-re...,diff --git a/language-adaptors/rxjava-groovy/s...,"[1, 5, 122, 9, 10, 9, 4, 1, 5, 18, 2, 0, 0, 0,...",forEach with Action1 but not Observer I re-rea...,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: lang/groovy/Ob...
4,ponsonio,RxJava,191f023cf5253ea90647bc091dcaf55ccdce81cc,Corrective,1.x: Fix Completable swallows- OnErrorNotImple...,diff --git a/src/main/java/rx/Completable.java...,"[1, 1, 0, 0, 0, 0, 0, 1, 21, 0, 0, 0, 0, 0, 0,...",1.x: Completable swallows OnErrorNotImplemente...,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: java/rx/Comple...
...,...,...,...,...,...,...,...,...,...,...
1776,jenkinsci,clearcase-plugin,51e9da224f80254476a7dc446bca817b505381d8,Perfective,Use a temporary file to decrease memory consum...,diff --git a/src/main/java/hudson/plugins/clea...,"[2, 12, 0, 4, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,...",Use a temporary file to decrease memory consum...,clearcase-plugin [Change] ChangeScribeStart\nS...,Summarized Code Changes:\nFile: plugins/clearc...
1777,jexp,batch-import,609d6c4b1eea2c33d9fb950fcbb9ba9dc1f80fc3,Perfective,added a more memory efficient structure for st...,diff --git a/src/main/java/org/neo4j/batchimpo...,"[10, 159, 29, 35, 9, 2, 1, 5, 106, 0, 4, 8, 0,...",added a more memory efficient structure for st...,batch-import [Change] ChangeScribeStart\nSumma...,Summarized Code Changes:\nFile: neo4j/batchimp...
1778,hdiv,hdiv,19b650c78a1c76f4fd90274d7f163f863c0d39e4,Perfective,Memory and performance optimizations,diff --git a/hdiv-config/src/main/java/org/hdi...,"[31, 302, 131, 140, 170, 89, 53, 7, 88, 14, 17...",Memory and performance optimizations,hdiv [Change] ChangeScribeStart\nSummarized Co...,Summarized Code Changes:\nFile: config/xml/Con...
1779,casidiablo,persistence,d7bf95159df37a3d338ca267dddd3d26b38ec37c,Perfective,Now it is possible to specify the sqlite open ...,diff --git a/pom.xml b/pom.xml\nindex 394263b....,"[5, 57, 20, 9, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Now it is possible to specify the sqlite open ...,persistence [Change] ChangeScribeStart\nSummar...,Summarized Code Changes:\nFile: pom.xml\n - Mo...
